In [ ]:
!git clone https://github.com/JAdilsonSA/-rvore-RN

In [ ]:
#no.py

class No:
    def __init__(self, chave):
        self.chave = chave
        self.cor = "VERMELHO"  # Todo novo nó nasce vermelho
        self.esquerda = None
        self.direita = None
        self.pai = None

In [ ]:
# arvore_rubro_negra.py


class ArvoreRubroNegra:

    def __init__(self):

      # Nó sentinela NIL utilizado em vez de None.
      # Todo NIL é considerado preto.
      self.NIL = No(None)
      self.NIL.cor = "PRETO"

      self.NIL.esquerda = self.NIL
      self.NIL.direita = self.NIL

      #Árvore começa vazia
      self.raiz = self.NIL

    # Rotação à esquerda
    def rotacao_esquerda(self, x):  # Rotação utilizada para restaurar o balanceamento
                                    # após inserções ou remoções.
        y = x.direita
        x.direita = y.esquerda

        if y.esquerda:
            y.esquerda.pai = x

        y.pai = x.pai

        if not x.pai:
            self.raiz = y

        elif x == x.pai.esquerda:
            x.pai.esquerda = y

        else:
            x.pai.direita = y

        y.esquerda = x
        x.pai = y

    # Rotação à direita
    def rotacao_direita(self, y):   # Rotação simétrica da rotação à esquerda.
                                    # Mantém a propriedade de busca da árvore.
        x = y.esquerda
        y.esquerda = x.direita

        if x.direita:
            x.direita.pai = y

        x.pai = y.pai

        if not y.pai:
            self.raiz = x

        elif y == y.pai.direita:
            y.pai.direita = x

        else:
            y.pai.esquerda = x

        x.direita = y
        y.pai = x

    # Inserção simples
    def inserir(self, chave):  # Cria um novo nó vermelho.
                               # Em árvores rubro-negras todo nó nasce vermelho.
        novo = No(chave)

        novo.esquerda = self.NIL
        novo.direita = self.NIL

        pai = None
        atual = self.raiz

        while atual != self.NIL:    # Busca a posição correta para inserção, seguindo as regras da árvore binária de busca.
            pai = atual

            if novo.chave < atual.chave:
                atual = atual.esquerda
            else:
                atual = atual.direita

        novo.pai = pai

        if pai is None:
            self.raiz = novo

        elif novo.chave < pai.chave:
            pai.esquerda = novo

        else:
            pai.direita = novo

        self.corrigir_insercao(novo)   # Corrige possíveis violações das propriedadesc rubro-negras causadas pela inserção.

    # Correção das propriedades rubro-negras
    def corrigir_insercao(self, z):

        while z != self.raiz and z.pai.cor == "VERMELHO": # Enquanto o pai for vermelho existe violação da regra que proíbe dois nós vermelhos consecutivos.

            # Caso em que o pai está à esquerda do avô
            if z.pai == z.pai.pai.esquerda:

                # Identifica o tio
                tio = z.pai.pai.direita

                # Caso 1: tio vermelho
                if tio and tio.cor == "VERMELHO":

                    #Recoloração
                    z.pai.cor = "PRETO"
                    tio.cor = "PRETO"
                    z.pai.pai.cor = "VERMELHO"

                    # Continua verificando a partir do avô
                    z = z.pai.pai

                else:

                    # Caso 2: nó é filho direito
                    if z == z.pai.direita:
                        z = z.pai
                        self.rotacao_esquerda(z)

                    # Caso 3: rotação e recoloração
                    z.pai.cor = "PRETO"
                    z.pai.pai.cor = "VERMELHO"

                    self.rotacao_direita(z.pai.pai)

            # Caso simétrico: pai está à direita do avô
            else:

                tio = z.pai.pai.esquerda

                # Caso 1: tio vermelho
                if tio and tio.cor == "VERMELHO":

                    z.pai.cor = "PRETO"
                    tio.cor = "PRETO"
                    z.pai.pai.cor = "VERMELHO"

                    z = z.pai.pai

                else:

                    # Caso 2: nó é filho esquerdo
                    if z == z.pai.esquerda:
                        z = z.pai
                        self.rotacao_direita(z)

                    # Caso 3: rotação e recoloração
                    z.pai.cor = "PRETO"
                    z.pai.pai.cor = "VERMELHO"

                    self.rotacao_esquerda(z.pai.pai)

        # Garante que a raiz seja sempre preta
        self.raiz.cor = "PRETO"

    # Busca um valor na árvore
    def buscar(self, chave):

      atual = self.raiz

      while atual != self.NIL:

        if chave == atual.chave:
            return atual

        elif chave < atual.chave:
            atual = atual.esquerda

        else:
            atual = atual.direita

      return self.NIL

    # Remove um valor da árvore
    def remover(self, chave):

      # Localiza o nó que será removido.
      z = self.buscar(chave)

      if z == self.NIL:
        return False

      y = z

      # Armazena a cor do nó removido.
      # Se um nó preto for removido pode ser necessário
      # corrigir a árvore.
      cor_original = y.cor

      if z.esquerda == self.NIL: # Caso 1: Nó possui apenas filho direito ou nenhum filho.

        x = z.direita

        self.substituir(z, z.direita)

      elif z.direita == self.NIL: # Caso 2: Nó possui apenas filho esquerdo.

        x = z.esquerda

        self.substituir(z, z.esquerda)

      else: # Caso 3: Nó possui dois filhos. Utiliza o sucessor em ordem.

        y = self.minimo(z.direita)

        cor_original = y.cor

        x = y.direita

        if y.pai == z:

            x.pai = y

        else:

            self.substituir(y, y.direita)

            y.direita = z.direita

            y.direita.pai = y

        self.substituir(z, y)

        y.esquerda = z.esquerda

        y.esquerda.pai = y

        y.cor = z.cor

      if cor_original == "PRETO":
        self.corrigir_remocao(x)

      return True

    def corrigir_remocao(self, x):

      while x != self.raiz and x.cor == "PRETO":

        if x == x.pai.esquerda:

            irmao = x.pai.direita

            # Caso 1: Irmão vermelho.
            if irmao.cor == "VERMELHO":

                irmao.cor = "PRETO"
                x.pai.cor = "VERMELHO"

                self.rotacao_esquerda(x.pai)

                irmao = x.pai.direita

            # Caso 2: Irmão preto com filhos pretos.
            if (irmao.esquerda.cor == "PRETO" and
                irmao.direita.cor == "PRETO"):

                irmao.cor = "VERMELHO"

                x = x.pai

            else:

                # Caso 3: Irmão preto com filho interno vermelho.
                if irmao.direita.cor == "PRETO":

                    irmao.esquerda.cor = "PRETO"

                    irmao.cor = "VERMELHO"

                    self.rotacao_direita(irmao)

                    irmao = x.pai.direita

                # Caso 4: Irmão preto com filho externo vermelho.
                irmao.cor = x.pai.cor

                x.pai.cor = "PRETO"

                irmao.direita.cor = "PRETO"

                self.rotacao_esquerda(x.pai)

                x = self.raiz

        else:

            irmao = x.pai.esquerda

            # Espelho do caso anterior
            if irmao.cor == "VERMELHO":

                irmao.cor = "PRETO"

                x.pai.cor = "VERMELHO"

                self.rotacao_direita(x.pai)

                irmao = x.pai.esquerda

            if (irmao.direita.cor == "PRETO" and
                irmao.esquerda.cor == "PRETO"):

                irmao.cor = "VERMELHO"

                x = x.pai

            else:

                if irmao.esquerda.cor == "PRETO":

                    irmao.direita.cor = "PRETO"

                    irmao.cor = "VERMELHO"

                    self.rotacao_esquerda(irmao)

                    irmao = x.pai.esquerda

                irmao.cor = x.pai.cor

                x.pai.cor = "PRETO"

                irmao.esquerda.cor = "PRETO"

                self.rotacao_direita(x.pai)

                x = self.raiz

      x.cor = "PRETO"

    # Substitui um nó por outro
    def substituir(self, antigo, novo):

      if antigo.pai is None:
        self.raiz = novo

      elif antigo == antigo.pai.esquerda:
        antigo.pai.esquerda = novo

      else:
        antigo.pai.direita = novo

      novo.pai = antigo.pai

    # Retorna o menor nó de uma subárvore
    def minimo(self, no):

      while no.esquerda != self.NIL:
        no = no.esquerda

      return no

    # Percurso em ordem Crescente
    def em_ordem(self, no):

        if no:
            self.em_ordem(no.esquerda)
            print(f"{no.chave} ({no.cor})")
            self.em_ordem(no.direita)


    def mostrar_arvore(self, no, espaco="", ultimo=True):

        if no != self.NIL:

            print(espaco, end="")

            if ultimo:
                print("└── ", end="")
                novo_espaco = espaco + "    "
            else:
                print("├── ", end="")
                novo_espaco = espaco + "│   "

            print(f"{no.chave} ({no.cor[0]})")

            self.mostrar_arvore(no.esquerda, novo_espaco, False)
            self.mostrar_arvore(no.direita, novo_espaco, True)


In [ ]:
def menu():
    print("\n===== ÁRVORE RUBRO-NEGRA =====")
    print("1 - Inserir valor")
    print("2 - Buscar valor")
    print("3 - Remover valor")
    print("4 - Mostrar árvore")
    print("0 - Sair")


def main():

    arvore = ArvoreRubroNegra()

    # Valores iniciais para teste
    valores = [10, 20, 30, 15, 5, 25]

    for valor in valores:
        arvore.inserir(valor)

    print("Árvore criada com os valores:")
    print(valores)

    while True:

        menu()

        opcao = input("Escolha uma opção: ")

        if opcao == "1":

            valor = int(input("Valor a inserir: "))

            arvore.inserir(valor)

            print(f"{valor} inserido com sucesso!")

        elif opcao == "2":

            valor = int(input("Valor a buscar: "))

            resultado = arvore.buscar(valor)

            if resultado != arvore.NIL:

                print("\nValor encontrado!")
                print(f"Chave: {resultado.chave}")
                print(f"Cor: {resultado.cor}")

                if resultado.pai:
                    print(f"Pai: {resultado.pai.chave}")
                else:
                    print("Pai: None")

            else:
                print("\nValor não encontrado.")

        elif opcao == "3":

            valor = int(input("Valor a remover: "))

            if arvore.remover(valor):
                print(f"{valor} removido com sucesso!")
            else:
                print("Valor não encontrado.")

        elif opcao == "4":

            print("\nEstrutura da árvore:\n")

            if arvore.raiz != arvore.NIL:
                arvore.mostrar_arvore(arvore.raiz)
            else:
                print("Árvore vazia.")

        elif opcao == "0":

            print("Encerrando...")
            break

        else:
            print("Opção inválida!")


if __name__ == "__main__":
    main()

Árvore criada com os valores:
[10, 20, 30, 15, 5, 25]

===== ÁRVORE RUBRO-NEGRA =====
1 - Inserir valor
2 - Buscar valor
3 - Remover valor
4 - Mostrar árvore
0 - Sair
Escolha uma opção: 1
Valor a inserir: 10
10 inserido com sucesso!

===== ÁRVORE RUBRO-NEGRA =====
1 - Inserir valor
2 - Buscar valor
3 - Remover valor
4 - Mostrar árvore
0 - Sair
Escolha uma opção: 4

Estrutura da árvore:

└── 20 (P)
    ├── 10 (V)
    │   ├── 5 (P)
    │   └── 15 (P)
    │       ├── 10 (V)
    └── 30 (P)
        ├── 25 (V)

===== ÁRVORE RUBRO-NEGRA =====
1 - Inserir valor
2 - Buscar valor
3 - Remover valor
4 - Mostrar árvore
0 - Sair
Escolha uma opção: 0
Encerrando...
